In [0]:
%sql
-- 1. Drop Target Test/Silver Tables
DROP TABLE IF EXISTS healthone_lakehouse.test.hospitals;
DROP TABLE IF EXISTS healthone_lakehouse.silver.hospitals;

-- 2. Drop all Bronze Tables
DROP TABLE IF EXISTS healthone_lakehouse.bronze.hospitals;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.departments;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.employees;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.payroll;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.patients;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.doctors;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.appointments;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.admissions;
DROP TABLE IF EXISTS healthone_lakehouse.bronze.billing;

-- 3. Drop all Schemas (Now it is safe, tables are gone)
DROP SCHEMA IF EXISTS healthone_lakehouse.test;
DROP SCHEMA IF EXISTS healthone_lakehouse.silver;
DROP SCHEMA IF EXISTS healthone_lakehouse.gold;
DROP SCHEMA IF EXISTS healthone_lakehouse.metadata;
DROP SCHEMA IF EXISTS healthone_lakehouse.bronze;

-- 4. Drop the Catalog itself
DROP CATALOG IF EXISTS healthone_lakehouse;
-- Force drop the Catalog (removes any remaining schemas, even system ones)
DROP CATALOG IF EXISTS healthone_lakehouse CASCADE;

In [0]:
%sql
-- Force drop the Catalog (removes any remaining schemas, even system ones)
DROP CATALOG IF EXISTS healthone_lakehouse CASCADE;


In [0]:
# 1. Delete Auto Loader checkpoints
dbutils.fs.rm("abfss://test@sthealthonelakehouseci.dfs.core.windows.net/checkpoints/", recurse=True)

# 2. Delete Test Silver data
dbutils.fs.rm("abfss://test@sthealthonelakehouseci.dfs.core.windows.net/testing/hospitals/", recurse=True)
dbutils.fs.rm("abfss://test@sthealthonelakehouseci.dfs.core.windows.net/delta/hospitals/", recurse=True)

print("All physical data deleted.")

In [0]:
%sql
SELECT Year, Month, Day, COUNT(*) AS cnt
FROM healthone_lakehouse.bronze.hospitals
GROUP BY Year, Month, Day
ORDER BY Year, Month, Day;

In [0]:
# Remove the space at the beginning
raw_path = "abfss://test@sthealthonelakehouseci.dfs.core.windows.net/testing/hospitals/"
df = spark.read.format("delta").load(raw_path)

display(df.limit(20).orderBy("hospital_id"))
df.printSchema()
print("Number of rows:", df.count())